In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q11/q11_rewrite/checkpoints/pre_cell_3.pickle

In [4]:
%%RecordEvent
%%time
### cell 3 ###

# cell 3 optimized
# combine selection, computation, joins, and aggregation into a single GPU pipeline
df = (
    partsupp[['PS_PARTKEY','PS_SUPPKEY','PS_SUPPLYCOST','PS_AVAILQTY']]
    .assign(TOTAL_COST=lambda df: df.PS_SUPPLYCOST * df.PS_AVAILQTY)
    .merge(
        supplier[['S_SUPPKEY','S_NATIONKEY']],
        left_on='PS_SUPPKEY', right_on='S_SUPPKEY'
    )
    .merge(
        nation[nation.N_NAME == 'GERMANY'][['N_NATIONKEY']],
        left_on='S_NATIONKEY', right_on='N_NATIONKEY'
    )
)
# compute threshold on GPU
sum_val = float(df.TOTAL_COST.sum()) * 0.0001
# group, sum and filter on GPU
total = (
    df.groupby('PS_PARTKEY', sort=False)
      .TOTAL_COST.sum()
      .reset_index(name='VALUE')
)
total = total[total.VALUE > sum_val].sort_values('VALUE', ascending=False)

CPU times: user 60 ms, sys: 48.2 ms, total: 108 ms
Wall time: 119 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q11/rewritten/o4_mini_high_small/checkpoints/post_cell_3_try_1.pickle